# QUANTUM LAB — Partícula en una esfera · armónicos esféricos

Momento angular cuantizado, orbitales en 3D (reales y complejos), superficies nodales, energía de rotación del rotor rígido y partícula confinada dentro de una esfera.

**Criterio del curso que cumple este cuaderno**

- *Densidad de probabilidad (armónicos esféricos) frente a la posición en coordenadas cartesianas, para cualquier estado.*
  En el panel **«Estado de la partícula»** se escriben **l** y **mₗ**; la pestaña **«Densidad de probabilidad»** evalúa
  |Y_l^mₗ|² en el punto (x, y, z) que se elija (y en los seis puntos de los ejes) y la grafica: la esfera de radio R
  coloreada por |Y|²/R² en x, y, z; la superficie polar r = |Y|²; los cortes en los planos xz, yz y xy; y |Y|² frente a z.

**Cómo usarlo en Google Colab**

1. Menú **Entorno de ejecución → Ejecutar todo** (o pulsa ▶ en cada celda, en orden). No hace falta instalar nada.
2. Al final aparece el simulador: los **parámetros** arriba y los resultados organizados en **pestañas**.
3. Escribe **cualquier valor** en las casillas y pulsa *Enter* (o mueve el deslizador, que amplía su rango solo).
   Se admiten formas como `2.5`, `2,5`, `1e-30`, `3*10^8` o `2*pi`.
4. Si un valor no tiene sentido físico (masa negativa, n = 2.5, una caja más pequeña que la longitud de Planck…),
   el simulador **no se detiene**: el apartado **«VALIDEZ FÍSICA DE LOS DATOS»** explica qué ocurre y por qué.
   - **[ERROR]**: con ese dato no se puede calcular (se explica el motivo físico).
   - **[AVISO]**: se calcula, pero el resultado no tiene respaldo físico (por ejemplo, el modelo deja de valer).
   - **[NOTA]**: información útil para interpretar el resultado.
5. Cada resultado se presenta como en los simuladores de Bohr y del efecto fotoeléctrico: parámetros, validez física,
   resultados alineados, tablas, interpretación física y gráficas.

Para ver el código de una celda, haz doble clic sobre su título.


In [ ]:
#@title 1) Utilidades comunes: formato de resultados, entradas sin límite y validez física { display-mode: "form" }
# =====================================================================================
#  Esta celda no contiene física: solo da formato a los resultados (recuadros, columnas
#  alineadas, tablas), lee los valores escritos por el usuario y explica los datos que
#  no tienen sentido físico. Ejecútala antes que las demás.
# =====================================================================================
import ast
import math
import operator
import textwrap
import warnings
import html as _html

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as w
from IPython.display import display, clear_output, HTML
from scipy import constants as const

warnings.filterwarnings("ignore", message=".*IPython.display.IFrame.*")
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ------------------------------------------------------------------
# Constantes físicas (CODATA, vía scipy.constants)
# ------------------------------------------------------------------
HBAR = const.hbar                    # J·s
H_PLANCK = const.h                   # J·s
C_LUZ = const.c                      # m/s
M_E = const.m_e                      # kg
EV = const.e                         # J por eV
K_B = const.k                        # J/K
A0 = const.physical_constants["Bohr radius"][0]          # m
E_H = const.physical_constants["Hartree energy"][0]      # J
L_PLANCK = const.physical_constants["Planck length"][0]  # m
KT_AMBIENTE_EV = K_B * 300 / EV      # energía térmica a 300 K  ≈ 0.0259 eV
ME_C2_EV = M_E * C_LUZ ** 2 / EV     # energía en reposo del electrón ≈ 511 keV

# ------------------------------------------------------------------
# Formato numérico
# ------------------------------------------------------------------
ANCHO = 80


def num(x, cifras=6):
    """Número legible: decimal si es «normal», notación científica si es muy grande o muy pequeño."""
    if x is None:
        return "—"
    try:
        x = float(x)
    except (TypeError, ValueError):
        return str(x)
    if math.isnan(x):
        return "no definido"
    if math.isinf(x):
        return "infinito" if x > 0 else "−infinito"
    if x == 0:
        return "0"
    if 1e-3 <= abs(x) < 1e6:
        texto = f"{x:.{cifras}g}"
    else:
        texto = f"{x:.{max(cifras - 2, 2)}e}"
    return texto.replace("-", "−")


def num_log(log10_valor, cifras=4):
    """Escribe 10^(log10_valor) aunque el número no quepa en un float (por ejemplo 10^5000)."""
    if not np.isfinite(log10_valor):
        return "0" if log10_valor < 0 else "infinito"
    exponente = math.floor(log10_valor)
    mantisa = 10 ** (log10_valor - exponente)
    if mantisa >= 9.9999999:
        mantisa, exponente = 1.0, exponente + 1
    if -3 <= exponente < 6:
        return num(10 ** log10_valor, cifras + 2)
    return f"{mantisa:.{cifras}f}e{exponente:+03d}".replace("-", "−")


def entero(n):
    """Entero con separador de miles (espacio fino) para leer números enormes."""
    return f"{int(n):,}".replace(",", " ")


_ESCALAS_LONGITUD = [(1e-15, "fm"), (1e-12, "pm"), (1e-9, "nm"), (1e-6, "µm"), (1e-3, "mm"), (1.0, "m"), (1e3, "km")]


def escala_longitud(L_m):
    """Unidad cómoda para dibujar una longitud L (en metros): devuelve (factor, nombre)."""
    for factor, nombre in reversed(_ESCALAS_LONGITUD):
        if L_m >= 0.1 * factor:
            return factor, nombre
    return 1e-15, "fm"


# ------------------------------------------------------------------
# Impresión con el estilo de QUANTUM LAB (recuadros y columnas alineadas)
# ------------------------------------------------------------------
def titulo_caja(titulo):
    print()
    print("╔" + "═" * (ANCHO - 2) + "╗")
    print("║" + titulo.center(ANCHO - 2) + "║")
    print("╚" + "═" * (ANCHO - 2) + "╝")


def seccion(titulo):
    print()
    print("─" * ANCHO)
    print(titulo)
    print("─" * ANCHO)


def dato(etiqueta, valor, unidad="", ancho_etiqueta=40):
    texto = f"{valor} {unidad}".rstrip()
    print(f"   {etiqueta:<{ancho_etiqueta}}: {texto}")


def parrafo(texto, sangria=3):
    for linea in textwrap.wrap(texto, ANCHO - sangria):
        print(" " * sangria + linea)


def vinetas(lineas, sangria=3):
    for linea in lineas:
        partes = textwrap.wrap(linea, ANCHO - sangria - 2)
        for i, parte in enumerate(partes):
            print(" " * sangria + ("• " if i == 0 else "  ") + parte)


def tabla(encabezados, filas, sangria=5, alinear=None):
    """Tabla con bordes de caja; cada columna se ajusta a su contenido. alinear: «c» centrado, «l» izquierda."""
    filas = [[str(c) for c in f] for f in filas]
    anchos = [max(len(str(h)), *(len(f[i]) for f in filas)) + 4 if filas else len(str(h)) + 4
              for i, h in enumerate(encabezados)]
    margen = " " * sangria

    def linea(izq, cruce, der):
        return margen + izq + cruce.join("─" * a for a in anchos) + der

    print(linea("┌", "┬", "┐"))
    alinear = alinear or "c" * len(encabezados)

    def celda(c, a, modo):
        return (" " + c).ljust(a) if modo == "l" else c.center(a)

    print(margen + "│" + "│".join(str(h).center(a) for h, a in zip(encabezados, anchos)) + "│")
    print(linea("├", "┼", "┤"))
    for f in filas:
        print(margen + "│" + "│".join(celda(c, a, m) for c, a, m in zip(f, anchos, alinear)) + "│")
    print(linea("└", "┴", "┘"))


# ------------------------------------------------------------------
# Validez física de los datos
# ------------------------------------------------------------------
class Diagnostico:
    """
    Reúne los comentarios sobre los datos introducidos:
      ERROR  no se puede calcular (por ejemplo, una masa negativa o n = 2.5);
      AVISO  se calcula, pero el resultado no tiene respaldo físico o el modelo deja de valer;
      NOTA   información útil para interpretar los resultados.
    """

    def __init__(self):
        self.items = []

    def error(self, texto):
        self.items.append(("ERROR", texto))

    def aviso(self, texto):
        self.items.append(("AVISO", texto))

    def nota(self, texto):
        self.items.append(("NOTA", texto))

    @property
    def hay_error(self):
        return any(nivel == "ERROR" for nivel, _ in self.items)

    def imprimir(self):
        titulo_caja("VALIDEZ FÍSICA DE LOS DATOS")
        if not self.items:
            print("   Todos los valores están dentro del rango de validez del modelo.")
            return
        orden = {"ERROR": 0, "AVISO": 1, "NOTA": 2}
        for nivel, texto in sorted(self.items, key=lambda it: orden[it[0]]):
            etiqueta = f"   [{nivel}] "
            partes = textwrap.wrap(texto, ANCHO - len(etiqueta))
            for i, parte in enumerate(partes):
                print((etiqueta if i == 0 else " " * len(etiqueta)) + parte)
            print()
        if self.hay_error:
            print("   >> Con estos datos no se puede calcular. Corrige los valores marcados como ERROR.")


def revisar_masa(m_me, diag, nombre="La masa"):
    """Comprueba una masa expresada en masas del electrón. Devuelve la masa en kg o None."""
    if m_me is None:
        return None
    if m_me == 0:
        diag.error(f"{nombre} vale 0. Una partícula sin masa (como el fotón) no obedece la ecuación de "
                   "Schrödinger: siempre viaja a la velocidad de la luz y necesita una descripción relativista. "
                   "Además, todas las energías del modelo serían infinitas (E ∝ 1/m).")
        return None
    if m_me < 0:
        diag.error(f"{nombre} es negativa ({num(m_me)} mₑ). No existe materia con masa negativa: la energía "
                   "cinética p²/2m saldría negativa y los estados ligados no tendrían sentido.")
        return None
    if m_me < 1e-6:
        diag.aviso(f"{nombre} ({num(m_me)} mₑ = {num(m_me * ME_C2_EV)} eV/c²) es menor que la de cualquier "
                   "partícula con masa conocida (los neutrinos tienen menos de ~1e−6 mₑ). Se calcula, pero no "
                   "corresponde a ninguna partícula real.")
    elif m_me > 1e15:
        diag.nota(f"{nombre} ({num(m_me * M_E)} kg) es la de un objeto macroscópico o casi. Su longitud de onda "
                  "de De Broglie es diminuta y los efectos cuánticos son imperceptibles: se comporta de forma "
                  "clásica (principio de correspondencia).")
    return m_me * M_E


def revisar_longitud(L_m, diag, nombre, que="longitud"):
    """Comentarios sobre una longitud física (en metros) ya validada como positiva."""
    if L_m < L_PLANCK:
        diag.aviso(f"{nombre} ({num(L_m)} m) es menor que la longitud de Planck (1.6e−35 m). A esa escala la "
                   "gravedad cuántica es importante y ninguna teoría física conocida es fiable: el resultado es "
                   "solo la evaluación matemática de la fórmula.")
    elif L_m < 1e-15:
        diag.aviso(f"{nombre} ({num(L_m)} m) es menor que un protón (≈ 1e−15 m). La partícula estaría confinada "
                   "en una región más pequeña que un núcleo: allí dominan la física nuclear y la relativista.")
    elif L_m > 1.0:
        diag.nota(f"{nombre} ({num(L_m)} m) es de tamaño macroscópico. Los niveles de energía quedan tan juntos que "
                  "el espectro es prácticamente continuo: es el límite clásico.")


def revisar_relatividad(log10_E_J, m_kg, diag, que="La energía"):
    """Compara una energía con la energía en reposo mc² (en escala logarítmica para no desbordar)."""
    log10_mc2 = math.log10(m_kg * C_LUZ ** 2)
    cociente = log10_E_J - log10_mc2
    if cociente > 0:
        diag.aviso(f"{que} es {num_log(cociente, 3)} veces la energía en reposo mc² de la partícula. Con tanta "
                   "energía se crearían pares partícula–antipartícula y la ecuación de Schrödinger (no relativista) "
                   "deja de valer: los números son el resultado matemático de la fórmula, no una predicción física.")
    elif cociente > -2:
        diag.aviso(f"{que} es el {num(100 * 10 ** cociente, 3)} % de mc²: la velocidad de la partícula es una "
                   "fracción apreciable de la de la luz y el modelo no relativista empieza a fallar.")


# ------------------------------------------------------------------
# Lectura segura de los valores escritos (admite 1e-30, 2,5, 2*pi, 3×10^8…)
# ------------------------------------------------------------------
_OPERACIONES = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
                ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos}
_NOMBRES = {"pi": math.pi, "e": math.e, "inf": math.inf, "infinito": math.inf}


def _evaluar(nodo):
    if isinstance(nodo, ast.Expression):
        return _evaluar(nodo.body)
    if isinstance(nodo, ast.Constant) and isinstance(nodo.value, (int, float)):
        return float(nodo.value)
    if isinstance(nodo, ast.Name) and nodo.id.lower() in _NOMBRES:
        return _NOMBRES[nodo.id.lower()]
    if isinstance(nodo, ast.BinOp) and type(nodo.op) in _OPERACIONES:
        a, b = _evaluar(nodo.left), _evaluar(nodo.right)
        if isinstance(nodo.op, ast.Pow) and abs(b) > 1e4 and abs(a) > 1:
            return math.inf if b > 0 else 0.0
        return _OPERACIONES[type(nodo.op)](a, b)
    if isinstance(nodo, ast.UnaryOp) and type(nodo.op) in _OPERACIONES:
        return _OPERACIONES[type(nodo.op)](_evaluar(nodo.operand))
    raise ValueError


def leer_numero(texto):
    """Convierte el texto escrito en un número. Devuelve (valor, None) o (None, explicación)."""
    t = str(texto).strip()
    if t == "":
        return None, "la casilla está vacía"
    t = (t.replace("π", "pi").replace("−", "-").replace("×", "*").replace("^", "**").replace(" ", ""))
    if t.count(",") == 1 and "." not in t:
        t = t.replace(",", ".")                          # coma decimal: 2,5 → 2.5
    t = t.replace("x10", "*10").replace("X10", "*10")
    try:
        valor = _evaluar(ast.parse(t, mode="eval"))
    except OverflowError:
        return None, (f"«{texto}» es mayor que 1.8e308, el número más grande que puede manejar el ordenador "
                      "(un float de 64 bits)")
    except ZeroDivisionError:
        return None, f"«{texto}» contiene una división entre cero"
    except (ValueError, SyntaxError, TypeError, RecursionError, MemoryError):
        return None, f"«{texto}» no es un número (ejemplos válidos: 2.5, 2,5, 1e-30, 3*10^8, 2*pi)"
    if isinstance(valor, complex):
        return None, f"«{texto}» da un número complejo"
    if math.isinf(valor) and "inf" not in t.lower():
        return None, (f"«{texto}» es mayor que 1.8e308, el número más grande que puede manejar el ordenador "
                      "(un float de 64 bits)")
    return float(valor), None


# ------------------------------------------------------------------
# Entradas: deslizador + casilla de texto, SIN límite de rango
# ------------------------------------------------------------------
_ESTILO = {"description_width": "0px"}


class Entrada:
    """
    Control con deslizador y casilla de texto enlazados. La casilla admite CUALQUIER valor: si se escribe
    uno fuera del rango del deslizador, el deslizador amplía su rango. El valor que se usa es el de la casilla.
    tipo: "real", "entero" o "log" (deslizador logarítmico, para masas, longitudes o frecuencias).
    """

    def __init__(self, etiqueta, valor, unidad="", tipo="real", rango=(0.0, 1.0), paso=None, ayuda=""):
        self.etiqueta, self.tipo = etiqueta, tipo
        self._sincronizando = False
        disposicion = w.Layout(width="290px")
        if tipo == "entero":
            self.deslizador = w.IntSlider(value=int(valor), min=int(rango[0]), max=int(rango[1]), step=1,
                                          readout=False, continuous_update=False, layout=disposicion, style=_ESTILO)
        elif tipo == "log":
            self.deslizador = w.FloatLogSlider(value=float(valor), base=10, min=rango[0], max=rango[1],
                                               step=paso or 0.05, readout=False, continuous_update=False,
                                               layout=disposicion, style=_ESTILO)
        else:
            self.deslizador = w.FloatSlider(value=float(valor), min=rango[0], max=rango[1],
                                            step=paso or (rango[1] - rango[0]) / 200, readout=False,
                                            continuous_update=False, layout=disposicion, style=_ESTILO)
        self._rango_original = (self.deslizador.min, self.deslizador.max)
        self.texto = w.Text(value=self._formato(valor), continuous_update=False, layout=w.Layout(width="150px"))
        etiqueta_html = w.HTML(f"<div style='padding-top:4px' title='{_html.escape(ayuda)}'>{_html.escape(etiqueta)}</div>",
                               layout=w.Layout(width="250px"))
        unidad_html = w.HTML(f"<div style='padding-top:4px;opacity:.75'>{_html.escape(unidad)}</div>",
                             layout=w.Layout(width="200px"))
        self.caja = w.HBox([etiqueta_html, self.deslizador, self.texto, unidad_html],
                           layout=w.Layout(align_items="center", margin="1px 0"))
        self.deslizador.observe(self._desde_deslizador, "value")
        self.texto.observe(self._desde_texto, "value")

    def _formato(self, v):
        if self.tipo == "entero":
            return str(int(round(v)))
        return num(v, 6).replace("−", "-")

    def _desde_deslizador(self, cambio):
        if self._sincronizando:
            return
        self._sincronizando = True
        self.texto.value = self._formato(cambio["new"])
        self._sincronizando = False

    def _desde_texto(self, cambio):
        if self._sincronizando:
            return
        v, _ = leer_numero(cambio["new"])
        if v is None or not math.isfinite(v):
            return
        self._sincronizando = True
        try:
            d = self.deslizador
            lo, hi = self._rango_original
            # Si el valor vuelve a caber en el rango inicial, el deslizador recupera ese rango (más cómodo de usar)
            if self.tipo == "log" and v > 0 and lo <= math.log10(v) <= hi:
                d.min, d.max = lo, hi
            elif self.tipo != "log" and lo <= v <= hi:
                d.min, d.max = lo, hi
            if self.tipo == "log":
                if v > 0:
                    e = math.log10(v)
                    if e < d.min:
                        d.min = math.floor(e)
                    if e > d.max:
                        d.max = math.ceil(e)
                    d.value = v
            elif self.tipo == "entero":
                if float(v).is_integer() and abs(v) < 2 ** 31:
                    vi = int(v)
                    if vi < d.min:
                        d.min = vi
                    if vi > d.max:
                        d.max = vi
                    d.value = vi
            else:
                if v < d.min:
                    d.min = v
                if v > d.max:
                    d.max = v
                d.value = v
        except Exception:
            pass
        finally:
            self._sincronizando = False

    def leer(self, diag):
        """Valor numérico de la casilla; si no se puede leer, añade un ERROR al diagnóstico y devuelve None."""
        v, problema = leer_numero(self.texto.value)
        if problema:
            diag.error(f"{self.etiqueta}: {problema}.")
            return None
        if not math.isfinite(v):
            diag.error(f"{self.etiqueta} es infinito. Ninguna magnitud física medible es infinita; "
                       "escribe un valor finito, por grande que sea.")
            return None
        return v

    def fijar(self, v):
        self.texto.value = self._formato(v)

    def al_cambiar(self, funcion):
        self.texto.observe(lambda _: funcion(), "value")


def revisar_entero(v, etiqueta, diag, explicacion):
    """Comprueba que un número cuántico sea entero; si no lo es, explica por qué debe serlo."""
    if v is None:
        return None
    if abs(v - round(v)) > 1e-9 * max(1.0, abs(v)):
        diag.error(f"{etiqueta} = {num(v)} no es entero. {explicacion}")
        return None
    if abs(v) > 1e300:
        diag.error(f"{etiqueta} = {num(v)} es demasiado grande para representarlo (supera 1e300).")
        return None
    return int(round(v))


# ------------------------------------------------------------------
# Paneles de la interfaz
# ------------------------------------------------------------------
_BORDE = "1px solid rgba(128,128,128,0.45)"


def panel(titulo, filas):
    cabecera = w.HTML(f"<div style='font:600 11px monospace;letter-spacing:.14em;text-transform:uppercase;"
                      f"opacity:.7;padding-bottom:3px;border-bottom:{_BORDE};margin-bottom:4px'>"
                      f"{_html.escape(titulo)}</div>")
    return w.VBox([cabecera] + list(filas), layout=w.Layout(border=_BORDE, padding="8px 12px", margin="6px 0"))


def encabezado(titulo, subtitulo):
    display(HTML(f"<div style='margin:4px 0 10px 0'><div style='font:600 11px monospace;letter-spacing:.2em;"
                 f"opacity:.65'>QUANTUM LAB</div><div style='font-size:22px;font-weight:600;margin:2px 0'>"
                 f"{_html.escape(titulo)}</div><div style='opacity:.75'>{_html.escape(subtitulo)}</div></div>"))


def ayuda_entradas():
    return w.HTML("<div style='opacity:.7;font-size:12px;margin:2px 0 4px 0'>Escribe cualquier valor en las casillas "
                  "(admite 1e-30, 2,5, 3*10^8, 2*pi) y pulsa Enter, o usa el deslizador. Si un valor no tiene "
                  "sentido físico, el apartado «VALIDEZ FÍSICA DE LOS DATOS» explica qué ocurre.</div>")


def selector(etiqueta, opciones, valor=None, ancho="290px"):
    menu = w.Dropdown(options=opciones, value=valor, layout=w.Layout(width=ancho), style=_ESTILO)
    fila = w.HBox([w.HTML(f"<div style='padding-top:4px'>{_html.escape(etiqueta)}</div>", layout=w.Layout(width="250px")),
                   menu], layout=w.Layout(align_items="center", margin="1px 0"))
    return menu, fila


def casilla(texto, valor=False):
    return w.Checkbox(value=valor, description=texto, indent=False, layout=w.Layout(width="auto", margin="0 18px 0 0"))


def boton(texto, funcion, estilo="primary"):
    b = w.Button(description=texto, button_style=estilo, layout=w.Layout(width="auto", margin="4px 8px 4px 0"))
    b.on_click(lambda _: funcion())
    return b


def ejecutar_en(salida, funcion):
    """Ejecuta un cálculo dentro de una salida; cualquier fallo imprevisto se explica en lugar de romper el programa."""
    with salida:
        clear_output(wait=True)
        try:
            funcion()
        except Exception as error:      # noqa: BLE001 — se informa al usuario
            titulo_caja("NO SE PUDO COMPLETAR EL CÁLCULO")
            parrafo(f"Ocurrió un problema numérico inesperado: {type(error).__name__}: {error}")
            parrafo("Suele deberse a valores tan extremos que superan la precisión del ordenador. Prueba con "
                    "valores menos extremos; el resto del simulador sigue funcionando.")


# ------------------------------------------------------------------
# Estilo de las gráficas (fondo claro, como los simuladores de Bohr y del efecto fotoeléctrico)
# ------------------------------------------------------------------
COLOR = {"onda": "#2471a3", "densidad": "#d35400", "verde": "#147a4b", "rojo": "#c0392b", "morado": "#8e44ad",
         "naranja": "#f39c12", "gris": "#7f8c8d", "oscuro": "#34495e", "turquesa": "#16a085", "azul_claro": "#5dade2"}


def estilo_graficas():
    plt.rcdefaults()
    plt.rcParams.update({
        "figure.dpi": 100, "savefig.dpi": 110, "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "bold",
        "axes.labelsize": 11.5, "axes.grid": True, "grid.linestyle": "--", "grid.alpha": 0.35,
        "legend.fontsize": 9.5, "legend.framealpha": 0.95, "axes.spines.top": False, "axes.spines.right": False,
    })


def recuadro(ax, texto, x=0.97, y=0.95, color="#f39c12", ha="right", va="top", fondo="#fef9e7", tam=9.5):
    """Recuadro informativo como los de las gráficas de Bohr y del efecto fotoeléctrico."""
    ax.text(x, y, texto, transform=ax.transAxes, ha=ha, va=va, fontsize=tam, linespacing=1.5, zorder=10,
            bbox=dict(boxstyle="round,pad=0.55", facecolor=fondo, edgecolor=color, linewidth=1.6, alpha=0.96))


def mostrar(fig):
    """Muestra una figura de Matplotlib o de Plotly dentro de la salida de un widget."""
    if hasattr(fig, "savefig"):
        display(fig)
        plt.close(fig)
    else:
        pagina = fig.to_html(include_plotlyjs="cdn", full_html=True, config={"displaylogo": False, "responsive": True})
        alto = int(fig.layout.height or 600) + 20
        display(HTML(f'<iframe srcdoc="{_html.escape(pagina, quote=True)}" '
                     f'style="width:100%;height:{alto}px;border:0;background:white"></iframe>'))


estilo_graficas()
print("Utilidades de QUANTUM LAB listas.")


In [ ]:
#@title 2) Física: armónicos esféricos, rotor rígido y partícula confinada en una esfera { display-mode: "form" }
# =====================================================================================
#  Armónicos esféricos (θ polar, φ azimutal, fase de Condon–Shortley):
#      Y_l^m(θ, φ) = P̄_l^m(cos θ)·e^{imφ},     L² Y = ħ²l(l+1) Y,     L_z Y = ħm Y,     |m| ≤ l
#  P̄_l^m (Legendre asociada normalizada) se calcula con una recurrencia ESTABLE con reescalado: vale para
#  l de miles sin desbordar.
#
#  Partícula sobre la superficie de una esfera de radio R (rotor rígido):
#      E_l = ħ²·l(l+1) / (2mR²) = ħ²·l(l+1) / (2I),   I = mR²,   degeneración 2l + 1
#
#  Partícula confinada DENTRO de una esfera (pozo esférico infinito):
#      ψ = j_l(x_{nl}·r/R)·Y_l^m,     E_{nl} = ħ²·x_{nl}² / (2mR²),     x_{nl} = n-ésimo cero de j_l
# =====================================================================================
from scipy import special
from scipy.optimize import brentq

LETRAS = "spdfghiklmnoqrtuvwxyz"
L_DIBUJO_MAX = 400          # por encima no se dibuja la superficie (la malla no resolvería los nodos)
RESOLUCION_MAX = 300
CEROS_NUMERICOS_MAX = 4000


def letra(l):
    return LETRAS[l] if l < len(LETRAS) else f"[l={l}]"


def legendre_normalizada(l, m, x):
    """P̄_l^m(x) para m ≥ 0, normalizada para que Y = P̄·e^{imφ} cumpla ∫|Y|²dΩ = 1 (incluye (−1)^m)."""
    x = np.asarray(x, dtype=float)
    s2 = np.clip(1 - x ** 2, 0, None)
    log_inicial = 0.5 * (math.log(2 * m + 1) - math.log(4 * math.pi)
                         + sum(math.log((2 * k - 1) / (2 * k)) for k in range(1, m + 1)))
    with np.errstate(divide="ignore"):
        log_base = log_inicial + (m / 2) * np.log(s2) if m > 0 else np.full_like(x, log_inicial)
    escala = log_base.copy()
    signo = (-1) ** m
    p_ant = np.zeros_like(x)
    p = np.full_like(x, float(signo))                        # P̄_m^m / e^{escala}
    if l == m:
        return _desescalar(p, escala)
    p_ant, p = p, x * math.sqrt(2 * m + 3) * p               # P̄_{m+1}^m
    for k in range(m + 2, l + 1):
        a = math.sqrt((4 * k * k - 1) / (k * k - m * m))
        b = math.sqrt(((k - 1) ** 2 - m * m) / (4 * (k - 1) ** 2 - 1))
        p_ant, p = p, a * (x * p - b * p_ant)
        grande = np.abs(p) > 1e150
        if grande.any():
            p[grande] /= 1e150
            p_ant[grande] /= 1e150
            escala[grande] += math.log(1e150)
    return _desescalar(p, escala)


def _desescalar(p, escala):
    out = np.zeros_like(p)
    ok = (p != 0) & np.isfinite(escala)
    out[ok] = np.sign(p[ok]) * np.exp(np.log(np.abs(p[ok])) + escala[ok])
    return out


def armonico(l, m, theta, phi):
    """Y_l^m complejo (cualquier signo de m)."""
    ma = abs(m)
    y = legendre_normalizada(l, ma, np.cos(theta)) * np.exp(1j * ma * np.asarray(phi))
    return y if m >= 0 else (-1) ** ma * np.conj(y)


def armonico_real(l, m, theta, phi):
    """Orbitales reales: m > 0 → √2·(−1)^m·Re Y_l^m (tipo cos mφ); m < 0 → √2·(−1)^m·Im Y_l^|m| (tipo sin |m|φ)."""
    ma = abs(m)
    if m == 0:
        return np.real(armonico(l, 0, theta, phi))
    y = armonico(l, ma, theta, phi)
    return math.sqrt(2) * (-1) ** ma * (np.real(y) if m > 0 else np.imag(y))


NOMBRES_REALES = {(1, 0): "p_z", (1, 1): "p_x", (1, -1): "p_y", (2, 0): "d_z²", (2, 1): "d_xz", (2, -1): "d_yz",
                  (2, 2): "d_x²−y²", (2, -2): "d_xy", (3, 0): "f_z³", (3, 1): "f_xz²", (3, -1): "f_yz²",
                  (3, 2): "f_z(x²−y²)", (3, -2): "f_xyz", (3, 3): "f_x(x²−3y²)", (3, -3): "f_y(3x²−y²)"}


def nombre_orbital(l, m, real):
    if l == 0:
        return "s"
    if real and (l, m) in NOMBRES_REALES:
        return NOMBRES_REALES[(l, m)]
    return f"{letra(l)} (m = {m})".replace("-", "−")


def campo(l, m, real, magnitud, theta, phi, R_nm=1.0):
    """Valor que se representa: «densidad» |Y|², «prob» |Y|²/R² (por área), «valor» (Y real con signo), «re», «im» o «fase»."""
    if real:
        y = armonico_real(l, m, theta, phi)
        return y ** 2 / (R_nm ** 2 if magnitud == "prob" else 1.0) if magnitud in ("densidad", "prob") else y
    y = armonico(l, m, theta, phi)
    return {"densidad": np.abs(y) ** 2, "prob": np.abs(y) ** 2 / R_nm ** 2, "re": np.real(y), "im": np.imag(y), "fase": np.angle(y),
            "valor": np.real(y)}[magnitud]


def malla_angular(n_theta):
    theta = np.linspace(0, math.pi, n_theta)
    phi = np.linspace(0, 2 * math.pi, 2 * n_theta - 1)
    return np.meshgrid(theta, phi, indexing="ij")


def normalizacion_numerica(l, m, n_theta=200):
    """∫|Y|² dΩ con Gauss–Legendre en cos θ (exacta para polinomios) y trapecio en φ."""
    x, w = np.polynomial.legendre.leggauss(max(n_theta, l + 2))
    th = np.arccos(x)
    ph = np.linspace(0, 2 * math.pi, 64, endpoint=False)
    TH, PH = np.meshgrid(th, ph, indexing="ij")
    y = armonico(l, m, TH, PH)
    return float(np.sum(w[:, None] * np.abs(y) ** 2) * (2 * math.pi / 64))


def nodos_latitud(l, m):
    """Ángulos θ (grados) de los conos nodales: ceros de P̄_l^|m| en (0, π). Son l − |m|."""
    if l - abs(m) <= 0:
        return np.array([])
    th = np.linspace(1e-6, math.pi - 1e-6, max(4000, 40 * l))
    p = legendre_normalizada(l, abs(m), np.cos(th))
    ceros = []
    for i in np.where(np.sign(p[:-1]) * np.sign(p[1:]) < 0)[0]:
        ceros.append(brentq(lambda t: legendre_normalizada(l, abs(m), np.array([math.cos(t)]))[0], th[i], th[i + 1]))
    return np.degrees(np.array(ceros))


def nodos_meridianos(m):
    """Planos meridianos nodales de los orbitales REALES: |m| planos (ángulos φ en grados, 0–180)."""
    ma = abs(m)
    if ma == 0:
        return np.array([])
    desfase = 90.0 / ma if m > 0 else 0.0                     # cos(mφ) = 0  o  sin(mφ) = 0
    return np.sort((desfase + 180.0 * np.arange(ma) / ma) % 180.0)


# ------------------------------------------------------------------
# Validación con explicación física
# ------------------------------------------------------------------
EXPLICACION_L = ("Para que la función de onda tenga un único valor en cada punto y sea finita en los polos, la "
                 "ecuación angular solo tiene solución si l es un entero ≥ 0. Por eso el momento angular está "
                 "cuantizado: L² = ħ²l(l+1).")
EXPLICACION_M = ("Al dar una vuelta completa (φ → φ + 2π) la función e^{imφ} debe volver a su valor: e^{i2πm} = 1 "
                 "solo si m es entero. Un m no entero daría una función de onda con dos valores en el mismo punto.")


def revisar_lm(l_valor, m_valor, diag):
    l = revisar_entero(l_valor, "l", diag, EXPLICACION_L)
    m = revisar_entero(m_valor, "m", diag, EXPLICACION_M)
    if l is None or m is None:
        return None, None
    if l < 0:
        l_eq = -l - 1
        diag.aviso(f"l = {l} es negativo. La ecuación solo depende de l(l+1), y l(l+1) = {l * (l + 1)} también se "
                   f"obtiene con l = {l_eq}: es el mismo estado físico. Por convenio se usa l = {l_eq} (l ≥ 0).")
        l = l_eq
    if abs(m) > l:
        diag.error(f"|m| = {abs(m)} es mayor que l = {l}. L_z = mħ no puede superar |L| = √(l(l+1))·ħ = "
                   f"{num(math.sqrt(l * (l + 1)), 5)}ħ: la proyección de un vector no puede ser mayor que el vector. "
                   f"Matemáticamente P_l^m ≡ 0 si |m| > l (no hay función de onda). Valores permitidos: "
                   f"m = −{l}, …, {l}.")
        return l, None
    if l > L_DIBUJO_MAX:
        diag.nota(f"Con l = {entero(l)} hay {entero(l)} superficies nodales; ninguna malla razonable las resuelve. Se dan "
                  "los valores exactos (L², L_z, nodos, energía) pero no la superficie 3D.")
    elif l > 60:
        diag.nota(f"Con l = {l} los lóbulos son muy finos: aumenta la resolución angular para verlos bien.")
    return l, m


def revisar_rotor(R_nm, m_me, diag):
    R_m = None
    if R_nm is not None:
        if R_nm == 0:
            diag.error("El radio de la esfera es 0: la partícula no tiene dónde moverse y las energías serían infinitas "
                       "(E ∝ 1/R²).")
        elif R_nm < 0:
            diag.error(f"El radio es negativo ({num(R_nm)} nm): un radio no puede ser negativo.")
        else:
            R_m = R_nm * 1e-9
    m_kg = revisar_masa(m_me, diag)
    if R_m is not None:
        revisar_longitud(R_m, diag, "El radio de la esfera")
    return R_m, m_kg


# ------------------------------------------------------------------
# Partícula confinada en una esfera: ceros de las funciones de Bessel esféricas
# ------------------------------------------------------------------
def cero_bessel(l, k):
    """k-ésimo cero positivo de j_l. Numérico (barrido + brentq) o asintótico de McMahon si l o k son enormes."""
    if l == 0:
        return k * math.pi, "exacto (kπ)"
    if k <= CEROS_NUMERICOS_MAX and l <= CEROS_NUMERICOS_MAX:
        paso = 0.05
        x = l + 0.5
        cuenta, anterior = 0, special.spherical_jn(l, x)
        while True:
            xs = x + paso * np.arange(1, 2001)
            ys = special.spherical_jn(l, xs)
            valores = np.concatenate([[anterior], ys])
            cambios = np.where(np.sign(valores[:-1]) * np.sign(valores[1:]) < 0)[0]
            if cuenta + len(cambios) >= k:
                i = cambios[k - cuenta - 1]
                a = x + paso * i
                return brentq(lambda t: special.spherical_jn(l, t), a, a + paso, xtol=1e-13), "numérico"
            cuenta += len(cambios)
            x, anterior = xs[-1], ys[-1]
    beta = (k + l / 2) * math.pi
    mu = 4 * (l + 0.5) ** 2
    return beta - (mu - 1) / (8 * beta) - 4 * (mu - 1) * (7 * mu - 31) / (3 * (8 * beta) ** 3), "asintótico (McMahon)"


# ------------------------------------------------------------------
# Densidad de probabilidad |Y_l^m|² en función de la posición (x, y, z)
# ------------------------------------------------------------------
#  Para una partícula sobre una esfera de radio R, la densidad solo depende de la DIRECCIÓN del punto:
#      cos θ = z / r,   tan φ = y / x,   r = √(x² + y² + z²)
#      P(θ, φ) = |Y_l^m(θ, φ)|²   [sr⁻¹]      (por unidad de área de la esfera: |Y|² / R²)
PLANOS = {"xz": "Plano xz (y = 0)", "yz": "Plano yz (x = 0)", "xy": "Plano xy (z = 0)"}


def densidad_probabilidad(l, m, theta, phi, real=False):
    """|Y_l^m|² del armónico complejo o, con real=True, el cuadrado del orbital real (pₓ, d_xy…). Unidades: sr⁻¹."""
    theta, phi = np.atleast_1d(np.asarray(theta, dtype=float)), np.atleast_1d(np.asarray(phi, dtype=float))
    if real and m != 0:
        return armonico_real(l, m, theta, phi) ** 2
    return np.abs(armonico(l, m, theta, phi)) ** 2


def a_esfericas(x, y, z):
    """(x, y, z) → (r, θ, φ) con θ ∈ [0, π] y φ ∈ [0, 2π)."""
    r = math.sqrt(x * x + y * y + z * z)
    if r == 0:
        return 0.0, None, None
    return r, math.acos(max(-1.0, min(1.0, z / r))), math.atan2(y, x) % (2 * math.pi)


def curva_densidad_plano(l, m, plano, real=False, n=1441):
    """Corte polar en un plano coordenado: puntos (h, v) a distancia |Y|² del origen en cada dirección del plano."""
    t = np.linspace(0.0, 2 * math.pi, n)
    if plano == "xy":
        dens = densidad_probabilidad(l, m, np.full_like(t, math.pi / 2), t, real)
        return dens * np.cos(t), dens * np.sin(t), dens
    phi0 = 0.0 if plano == "xz" else math.pi / 2
    theta = np.where(t <= math.pi, t, 2 * math.pi - t)
    phi = np.where(t <= math.pi, phi0, phi0 + math.pi)
    dens = densidad_probabilidad(l, m, theta, phi, real)
    h = dens * np.sin(theta) * (np.cos(phi) if plano == "xz" else np.sin(phi))
    return h, dens * np.cos(theta), dens


def meridiano_maximo(m, real):
    """Azimut φ donde el orbital real alcanza su máximo (|cos|m|φ| o |sin|m|φ| = 1); 0 para el armónico complejo."""
    return math.pi / (2 * abs(m)) if (real and m < 0) else 0.0


def densidad_frente_a_z(l, m, real=False, phi=0.0, n=801):
    """|Y|² sobre la esfera frente a la altura z/R = cos θ, a lo largo del meridiano φ."""
    z = np.linspace(-1.0, 1.0, n)
    return z, densidad_probabilidad(l, m, np.arccos(z), np.full_like(z, phi), real)


In [ ]:
#@title 3) Gráficas: superficie 3D, mapas angulares, comparaciones y partícula confinada { display-mode: "form" }
import plotly.graph_objects as go

MAGNITUDES = {"densidad": "Densidad angular", "prob": "Densidad de probabilidad",
              "valor": "Valor del orbital real (con signo)",
              "re": "Parte real Re Y", "im": "Parte imaginaria Im Y", "fase": "Fase arg Y"}
FORMAS = {"lobulos": "Lóbulos (r ∝ |valor|)", "esfera": "Esfera coloreada (r = R)",
          "suave": "Deformación suave r = R(1 + α·valor)"}


def _escala_colores(magnitud):
    if magnitud in ("densidad", "prob"):
        return "Viridis", "viridis"
    if magnitud == "fase":
        return "Twilight", "twilight"
    return "RdBu", "RdBu"


def geometria(valores, forma, R, alfa, magnitud):
    """Radio de cada punto: la forma es un recurso visual (no una superficie física)."""
    if magnitud == "fase":
        forma = "esfera"                                   # una fase no tiene «tamaño»
    mx = float(np.max(np.abs(valores))) or 1.0
    f = valores / mx
    if forma == "esfera":
        return np.full_like(valores, R, dtype=float), forma
    if forma == "suave":
        return R * (1 + alfa * f), forma
    return R * np.abs(f), forma


def figura_3d(l, m, real, magnitud, forma, alfa, n_theta, R_vis=1.0, R_nm=1.0):
    TH, PH = malla_angular(n_theta)
    val = campo(l, m, real, magnitud, TH, PH, R_nm)
    r, forma_ef = geometria(val, forma, R_vis, alfa, magnitud)
    x, y, z = r * np.sin(TH) * np.cos(PH), r * np.sin(TH) * np.sin(PH), r * np.cos(TH)
    escala, _ = _escala_colores(magnitud)
    if magnitud == "fase":
        cmin, cmax = -math.pi, math.pi
    elif magnitud in ("densidad", "prob"):
        cmin, cmax = 0.0, float(val.max())
    else:
        lim = float(np.max(np.abs(val))) or 1.0
        cmin, cmax = -lim, lim
    fig = go.Figure(go.Surface(x=x, y=y, z=z, surfacecolor=val, colorscale=escala, cmin=cmin, cmax=cmax,
                               colorbar=dict(title=dict(text={"fase": "rad", "densidad": "Densidad angular [sr⁻¹]", "prob": "Densidad de probabilidad [nm⁻²]"}.get(magnitud, "valor"), side="right"),
                                             thickness=14, len=0.7),
                               lighting=dict(ambient=0.75, diffuse=0.6, specular=0.15), hoverinfo="skip"))
    L = float(np.max(r)) * 1.15 or 1.0
    eje = dict(range=[-L, L], showbackground=True, backgroundcolor="#f4f6f8", gridcolor="#d5d8dc",
               zerolinecolor="#aab7b8", title="")
    nombre = ("real " if real else "complejo ") + nombre_orbital(l, m, real)
    fig.update_layout(template="plotly_white", height=620, margin=dict(l=0, r=0, t=70, b=0),
                      title=dict(text=f"{MAGNITUDES[magnitud]} · l = {l}, m = {m}", x=0.02),
                      scene=dict(xaxis=dict(eje, title="x"), yaxis=dict(eje, title="y"), zaxis=dict(eje, title="z"),
                                 aspectmode="cube", camera=dict(eye=dict(x=1.5, y=1.5, z=1.0))))
    return fig


def figura_mapas(l, m, real, magnitud, R_nm=1.0):
    """Mapa θ–φ del valor con las líneas nodales y corte polar en el plano xz."""
    n = int(min(max(181, 6 * l + 1), 1201))
    th = np.linspace(0, math.pi, n)
    ph = np.linspace(0, 2 * math.pi, 2 * n - 1)
    TH, PH = np.meshgrid(th, ph, indexing="ij")
    val = campo(l, m, real, magnitud, TH, PH, R_nm)
    _, cmap = _escala_colores(magnitud)
    fig = plt.figure(figsize=(13.5, 5.6), constrained_layout=True)
    ax1 = fig.add_subplot(1, 2, 1)
    if magnitud in ("densidad", "prob"):
        vmin, vmax = 0, val.max()
    elif magnitud == "fase":
        vmin, vmax = -math.pi, math.pi
    else:
        vmax = np.max(np.abs(val)) or 1
        vmin = -vmax
    im = ax1.imshow(val, origin="upper", aspect="auto", extent=[0, 360, 180, 0], cmap=cmap, vmin=vmin, vmax=vmax)
    for t in nodos_latitud(l, m):
        ax1.axhline(t, color="white", ls="--", lw=1.2)
    if real:
        for p in nodos_meridianos(m):
            for q in (p, p + 180):
                ax1.axvline(q, color="white", ls=":", lw=1.4)
    ax1.set_xlabel("φ (azimut) [°]")
    ax1.set_ylabel("θ [°]")
    ax1.set_title("Mapa sobre la esfera")
    ax1.set_xticks(range(0, 361, 60))
    ax1.set_yticks(range(0, 181, 30))
    ax1.grid(False)
    fig.colorbar(im, ax=ax1, shrink=0.9, label=MAGNITUDES[magnitud])
    ax2 = fig.add_subplot(1, 2, 2, projection="polar")
    t = np.linspace(0, 2 * math.pi, 1441)
    theta_pol = np.where(t <= math.pi, t, 2 * math.pi - t)          # corte en el plano xz (φ = 0 y φ = π)
    phi_pol = np.where(t <= math.pi, 0.0, math.pi)
    y = (armonico_real(l, m, theta_pol, phi_pol) if real else np.abs(armonico(l, m, theta_pol, phi_pol)))
    rr = np.abs(y)
    ax2.set_theta_zero_location("N")
    ax2.set_theta_direction(-1)
    if real:
        ax2.fill(t, np.where(y >= 0, rr, 0), color=COLOR["onda"], alpha=0.55, label="ψ > 0")
        ax2.fill(t, np.where(y < 0, rr, 0), color=COLOR["rojo"], alpha=0.55, label="ψ < 0")
    else:
        ax2.fill(t, rr, color=COLOR["morado"], alpha=0.5, label="|Y|")
    ax2.plot(t, rr, color="black", lw=0.8)
    ax2.set_title("Corte en el plano xz", pad=14)
    ax2.legend(loc="lower right", bbox_to_anchor=(1.15, -0.05))
    return fig


def _lobulo(ax, l, m, real, n=60, titulo=""):
    TH, PH = malla_angular(n)
    if real:
        y = armonico_real(l, m, TH, PH)
        r = np.abs(y)
        colores = plt.cm.RdBu(0.5 + 0.5 * y / (np.max(np.abs(y)) or 1))
    else:
        y = armonico(l, m, TH, PH)
        r = np.abs(y)
        colores = plt.cm.twilight((np.angle(y) + math.pi) / (2 * math.pi))
    r = r / (r.max() or 1)
    ax.plot_surface(r * np.sin(TH) * np.cos(PH), r * np.sin(TH) * np.sin(PH), r * np.cos(TH), facecolors=colores,
                    rstride=1, cstride=1, linewidth=0, antialiased=True, shade=False)
    ax.set_box_aspect((1, 1, 1))
    for f in (ax.set_xlim, ax.set_ylim, ax.set_zlim):
        f(-0.8, 0.8)
    ax.set_axis_off()
    ax.set_title(titulo, fontsize=11)


def figura_todos_m(l, real=True):
    ms = list(range(-l, l + 1))
    if len(ms) > 13:
        ms = sorted(set(int(round(v)) for v in np.linspace(-l, l, 13)))
    columnas = min(len(ms), 7)
    filas = math.ceil(len(ms) / columnas)
    fig = plt.figure(figsize=(2.3 * columnas, 2.6 * filas + 0.6), constrained_layout=True)
    for i, m in enumerate(ms, 1):
        ax = fig.add_subplot(filas, columnas, i, projection="3d")
        _lobulo(ax, l, m, real, n=50 if l < 20 else 80, titulo=f"m = {m}\n{nombre_orbital(l, m, real)}".replace("-", "−"))
    tipo = "reales (azul +, rojo −)" if real else "complejos (color = fase)"
    return fig


def figura_complejo_real(l, m):
    ma = abs(m)
    fig = plt.figure(figsize=(13.5, 4.2), constrained_layout=True)
    paneles = [(l, ma, False, f"Complejo Y_l^{{+{ma}}}\n(color = fase)"), (l, -ma, False, f"Complejo Y_l^{{−{ma}}}\n(color = fase)"),
               (l, ma, True, f"Real tipo cos({'' if ma == 1 else ma}φ)\n{nombre_orbital(l, ma, True)}"),
               (l, -ma, True, f"Real tipo sin({'' if ma == 1 else ma}φ)\n{nombre_orbital(l, -ma, True)}")]
    for i, (ll, mm, real, titulo) in enumerate(paneles, 1):
        ax = fig.add_subplot(1, 4, i, projection="3d")
        _lobulo(ax, ll, mm, real, n=60, titulo=titulo.replace("{", "").replace("}", "").replace("_l^", f"_{l}^"))
    return fig


def figura_confinada(n, l, x_nl, R_m, niveles):
    f, u = escala_longitud(R_m)
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14.5, 5.0), constrained_layout=True,
                                        gridspec_kw={"width_ratios": [1, 1, 0.9]})
    if x_nl < 4e4:
        r = np.linspace(1e-9, 1, int(min(max(1500, 30 * x_nl), 200000)))
        jl = special.spherical_jn(l, x_nl * r)
        norma = math.sqrt(2) / abs(special.spherical_jn(l + 1, x_nl))
        Rr = norma * jl                                   # en unidades de R^(−3/2)
        P = r ** 2 * Rr ** 2
        ax1.plot(r * R_m / f, Rr, color=COLOR["onda"], lw=2.2)
        ax1.axhline(0, color="black", lw=0.8, alpha=0.6)
        ax2.plot(r * R_m / f, P, color=COLOR["densidad"], lw=2.2)
        ax2.fill_between(r * R_m / f, P, color=COLOR["densidad"], alpha=0.18)
        for ax in (ax1, ax2):
            ax.axvline(R_m / f, color=COLOR["oscuro"], lw=3)
            ax.set_xlabel(f"distancia al centro r [{u}]")
        ax1.set_ylabel("Función radial [R$^{-3/2}$]")
        ax1.set_title("Función radial")
        ax2.set_ylabel("Densidad de probabilidad radial [1/R]")
        ax2.set_title("Densidad de probabilidad radial")
    else:
        for ax in (ax1, ax2):
            ax.text(0.5, 0.5, "La función radial oscila\ndemasiado rápido para dibujarla", ha="center", va="center",
                    transform=ax.transAxes, fontsize=12)
            ax.set_axis_off()
    # Diagrama de niveles (en unidades de ħ²/2mR²: E = x²)
    for (nn, ll, xx) in niveles:
        sel = (nn == n and ll == l)
        ax3.hlines(xx ** 2, ll - 0.35, ll + 0.35, color=COLOR["rojo"] if sel else COLOR["onda"], lw=3.5 if sel else 2)
        ax3.text(ll + 0.4, xx ** 2, f"{nn}{letra(ll)}", va="center", fontsize=9, fontweight="bold" if sel else "normal")
    if not any(nn == n and ll == l for nn, ll, _ in niveles):
        ax3.text(0.5, 0.97, f"(el estado {n}{letra(l)} está fuera de este diagrama: E = {num(x_nl ** 2, 4)})",
                 transform=ax3.transAxes, ha="center", va="top", fontsize=8.5)
    ax3.set_xticks(range(0, 5))
    ax3.set_xticklabels([f"l = {k}\n({letra(k)})" for k in range(0, 5)])
    ax3.set_ylabel("Energía [ħ²/(2mR²)]")
    ax3.set_title("Niveles de energía")
    ax3.set_xlim(-0.6, 4.9)
    return fig


def figura_densidad_cartesiana(l, m, real):
    """Densidad de probabilidad |Y_l^m|² frente a la posición en coordenadas cartesianas (cortes y altura z)."""
    fig, ejes = plt.subplots(2, 2, figsize=(13.5, 12), constrained_layout=True)
    ejes = ejes.ravel()
    letras = {"xz": ("x", "z"), "yz": ("y", "z"), "xy": ("x", "y")}
    escala = float(np.max(densidad_probabilidad(l, m, *malla_angular(61), real)))
    lim = 0.0
    for ax, plano in zip(ejes[:3], PLANOS):
        h, v, dens = curva_densidad_plano(l, m, plano, real)
        lim = max(lim, float(np.max(np.abs(np.concatenate([h, v])))))
        ax.fill(h, v, color=COLOR["densidad"], alpha=0.30, lw=0)
        ax.plot(h, v, color=COLOR["densidad"], lw=2.2)
        ax.axhline(0, color="black", lw=0.7)
        ax.axvline(0, color="black", lw=0.7)
        a, b = letras[plano]
        ax.set_xlabel(f"{a}   (distancia al origen = |Y|² [sr⁻¹])")
        ax.set_ylabel(b)
        ax.set_title(f"Plano {plano}")
        ax.set_aspect("equal")
        if float(np.max(dens)) < 1e-12 * escala:
            ax.text(0.5, 0.62, "Densidad nula", transform=ax.transAxes,
                    ha="center", va="center", fontsize=11, color=COLOR["gris"])
    lim = 1.12 * (lim or 1.0)
    for ax in ejes[:3]:
        ax.set_xlim(-lim, lim)
        ax.set_ylim(-lim, lim)
    ax = ejes[3]
    phi_max = meridiano_maximo(m, real)
    z, d0 = densidad_frente_a_z(l, m, real, phi_max)
    etiqueta = (f"meridiano de máxima densidad, φ = {math.degrees(phi_max):.4g}°" if (real and m != 0)
                else "cualquier meridiano (|Y|² no depende de φ)")
    ax.plot(z, d0, color=COLOR["densidad"], lw=2.4, label=etiqueta)
    for t in nodos_latitud(l, m):
        ax.axvline(math.cos(math.radians(t)), color=COLOR["gris"], ls=":", lw=1.3)
    ax.set_xlabel("posición z / R = cos θ")
    ax.set_ylabel("Densidad angular [sr⁻¹]")
    ax.set_title("Densidad angular frente a z")
    ax.set_xlim(-1, 1)
    ax.set_ylim(bottom=0)
    ax.legend(loc="upper center")
    tipo = "orbital real " + nombre_orbital(l, m, True) if (real and m != 0) else "armónico complejo"
    fig.suptitle(f"Densidad angular |Y|² [sr⁻¹] · l = {l}, mₗ = {m} · {tipo}".replace("-", "−"), fontsize=13, fontweight="bold")
    return fig


def figura_densidad_3d(l, m, real, R_nm, n_theta=90):
    """Izquierda: la esfera real de radio R coloreada por |Y|²/R². Derecha: superficie polar r = |Y|²(θ, φ)."""
    from plotly.subplots import make_subplots
    TH, PH = malla_angular(n_theta)
    dens = densidad_probabilidad(l, m, TH, PH, real).reshape(TH.shape)
    xs, ys, zs = R_nm * np.sin(TH) * np.cos(PH), R_nm * np.sin(TH) * np.sin(PH), R_nm * np.cos(TH)
    xp, yp, zp = dens * np.sin(TH) * np.cos(PH), dens * np.sin(TH) * np.sin(PH), dens * np.cos(TH)
    fig = make_subplots(rows=1, cols=2, specs=[[{"type": "scene"}, {"type": "scene"}]], horizontal_spacing=0.02,
                        subplot_titles=["Densidad de probabilidad  P = |Y|²/R²", "Densidad angular  |Y|²"])
    fig.add_trace(go.Surface(x=xs, y=ys, z=zs, surfacecolor=dens / R_nm ** 2, colorscale="Viridis", cmin=0,
                             colorbar=dict(title=dict(text="Densidad de probabilidad<br>[nm⁻²]", side="right"), x=0.40, len=0.75, thickness=12)),
                  row=1, col=1)
    fig.add_trace(go.Surface(x=xp, y=yp, z=zp, surfacecolor=dens, colorscale="Viridis", cmin=0,
                             colorbar=dict(title=dict(text="Densidad angular<br>[sr⁻¹]", side="right"), x=1.0, len=0.75, thickness=12)),
                  row=1, col=2)
    L = 1.15 * float(np.max(dens)) or 1.0
    eje = dict(showbackground=True, backgroundcolor="#f4f6f8", gridcolor="#d5d8dc", zerolinecolor="#aab7b8")
    fig.update_layout(template="plotly_white", height=600, margin=dict(l=0, r=0, t=80, b=0),
                      title=dict(text=f"l = {l}, mₗ = {m} · radio R = {R_nm:.4g} nm" + (" · orbital real" if real and m != 0 else ""), x=0.02),
                      scene=dict(xaxis=dict(eje, title="x [nm]", range=[-1.15 * R_nm, 1.15 * R_nm]),
                                 yaxis=dict(eje, title="y [nm]", range=[-1.15 * R_nm, 1.15 * R_nm]),
                                 zaxis=dict(eje, title="z [nm]", range=[-1.15 * R_nm, 1.15 * R_nm]), aspectmode="cube"),
                      scene2=dict(xaxis=dict(eje, title="x", range=[-L, L]), yaxis=dict(eje, title="y", range=[-L, L]),
                                  zaxis=dict(eje, title="z", range=[-L, L]), aspectmode="cube"))
    return fig


In [ ]:
#@title 4) Resultados en texto (mismo formato que los simuladores de Bohr y del efecto fotoeléctrico) { display-mode: "form" }
def _revisar_representacion(n_theta, alfa, forma, magnitud, real, diag):
    if n_theta is not None:
        nt = int(round(n_theta))
        if nt < 8:
            diag.aviso(f"Una resolución de {num(n_theta)} puntos en θ es demasiado baja para formar una superficie: "
                       "se usan 8.")
            nt = 8
        elif nt > RESOLUCION_MAX:
            diag.nota(f"Una resolución de {num(n_theta)} puntos bloquearía el navegador (la superficie tendría "
                      f"{entero(2 * nt * nt)} puntos): se usan {RESOLUCION_MAX}.")
            nt = RESOLUCION_MAX
        n_theta = nt
    if alfa is not None and forma == "suave":
        if alfa < 0:
            diag.nota("α negativa invierte la deformación (los valores positivos se hunden). Es solo un recurso "
                      "visual, sin significado físico.")
        if abs(alfa) > 1:
            diag.nota(f"Con |α| = {num(abs(alfa))} > 1 el radio r = R(1 + α·valor) se hace negativo en algunas "
                      "direcciones y la superficie se cruza consigo misma. Es solo una representación geométrica.")
    if real and magnitud in ("re", "im", "fase"):
        diag.nota("Los orbitales reales son funciones reales: no tienen parte imaginaria ni fase variable. Se "
                  "representa su valor con signo.")
        magnitud = "valor"
    if not real and magnitud == "valor":
        magnitud = "re"
    return n_theta, magnitud


def informe_armonico(l_valor, m_valor, real, magnitud, forma, n_theta, alfa, R_nm, m_me, diag=None, dibujar=True):
    diag = diag or Diagnostico()
    titulo_caja("QUANTUM LAB — PARTÍCULA EN UNA ESFERA · ARMÓNICO ESFÉRICO")
    l, m = revisar_lm(l_valor, m_valor, diag)
    R_m, m_kg = revisar_rotor(R_nm, m_me, diag)
    n_theta, magnitud = _revisar_representacion(n_theta, alfa, forma, magnitud, real, diag)
    seccion("PARÁMETROS DEL SISTEMA")
    dato("Números cuánticos l y m", f"l = {entero(l) if l is not None else num(l_valor)},  m = "
                                     f"{entero(m) if m is not None else num(m_valor)}".replace("-", "−"))
    dato("Tipo de orbital", "real (combinación de ±m)" if real else "complejo (L_z definido)")
    dato("Radio de la esfera R", num(R_nm), "nm")
    dato("Masa de la partícula", num(m_me), "mₑ")
    if diag.hay_error or l is None or m is None or R_m is None or m_kg is None or n_theta is None or alfa is None:
        diag.imprimir()
        return None
    log_E = math.log10(HBAR ** 2 / (2 * m_kg * R_m ** 2)) + (math.log10(l * (l + 1)) if l > 0 else -math.inf)
    if l > 0:
        revisar_relatividad(log_E, m_kg, diag, f"La energía de rotación del nivel l = {entero(l)}")
    diag.imprimir()

    seccion("MOMENTO ANGULAR")
    dato("Orbital", nombre_orbital(l, m, real) + f"   (letra {letra(l)})")
    dato("L² = ħ²·l(l+1)", num(l * (l + 1)), "ħ²")
    dato("|L| = ħ·√(l(l+1))", num(math.sqrt(l * (l + 1)), 6), f"ħ   = {num(math.sqrt(l * (l + 1)) * HBAR)} J·s")
    if real and m != 0:
        dato("L_z", "no definido", f"(mezcla m = +{entero(abs(m))} y m = −{entero(abs(m))})")
        dato("  valores posibles de L_z", f"±{num(abs(m))} ħ", "con probabilidad 1/2 cada uno;  ⟨L_z⟩ = 0")
        dato("  (para L_z = mħ definido)", "elige el tipo «Complejo»", "")
    else:
        dato("L_z = mħ", num(m), f"ħ   = {num(m * HBAR)} J·s")
    if l > 0 and not (real and m != 0):
        dato("Ángulo entre L y el eje z", f"{math.degrees(math.acos(max(-1, min(1, m / math.sqrt(l * (l + 1)))))):.4f}",
             "°   (cos θ = m/√(l(l+1)))")
    dato("Degeneración (valores de m)", entero(2 * l + 1))
    dato("Paridad (−1)^l", "par" if l % 2 == 0 else "impar")
    seccion("SUPERFICIES NODALES")
    lat = nodos_latitud(l, m) if l <= 3000 else None
    dato("Conos nodales (l − |m|)", entero(l - abs(m)))
    if lat is not None and 0 < len(lat) <= 10:
        dato("  ángulos θ de los conos", ", ".join(f"{t:.3f}" for t in lat), "°")
    if real:
        dato("Planos meridianos (|m|)", entero(abs(m)))
        if 0 < abs(m) <= 10:
            dato("  ángulos φ de los planos", ", ".join(f"{p:.2f}" for p in nodos_meridianos(m)), "°")
    else:
        dato("Planos meridianos", "0", "(|Y_l^m|² no depende de φ: simetría axial)")
    dato("Total de superficies nodales", entero(l - abs(m) + (abs(m) if real else 0)))
    if l <= 2000:
        dato("Normalización ∫|Y|² dΩ (numérica)", f"{normalizacion_numerica(l, m):.10f}", "(debe ser 1)")
    seccion("ENERGÍA DE ROTACIÓN (partícula sobre la superficie de la esfera)")
    I = m_kg * R_m ** 2
    dato("Momento de inercia I = mR²", num(I), "kg·m²")
    dato("Energía E_l = ħ²l(l+1)/(2I)", num_log(log_E - math.log10(EV)) if l > 0 else "0", "eV")
    dato("Cuanto de rotación ħ²/(2I)", num(HBAR ** 2 / (2 * I) / EV), "eV")
    tabla(["l", "letra", "E_l [eV]", "degeneración 2l+1"],
          [[("» " if k == l else "") + entero(k), letra(k), num_log(math.log10(HBAR ** 2 / (2 * I)) + math.log10(k * (k + 1))
                                                                     - math.log10(EV)) if k > 0 else "0", entero(2 * k + 1)]
           for k in range(max(0, l - 2), l + 3)])
    seccion("INTERPRETACIÓN FÍSICA")
    vinetas([
        "El momento angular está cuantizado: su módulo solo puede valer ħ√(l(l+1)) y su proyección sobre z, mħ.",
        "Como |m| < √(l(l+1)) (salvo l = 0), L nunca apunta exactamente según z: forma un cono alrededor del eje.",
        "Los orbitales complejos tienen L_z definido y |Y|² con simetría axial; los reales (pₓ, d_xy…) son "
        "combinaciones de m y −m, con lóbulos orientados pero sin L_z definido.",
        f"Este orbital tiene {entero(l - abs(m) + (abs(m) if real else 0))} superficies nodales, donde la probabilidad "
        "es cero (en los orbitales reales siempre son l; en los complejos, l − |m|).",
        "La forma de «lóbulos» es una representación: la distancia al origen indica |Y| en esa dirección, no una "
        "superficie física.",
    ])
    if dibujar and l <= L_DIBUJO_MAX:
        mostrar(figura_3d(l, m, real, magnitud, forma, alfa, n_theta, R_nm=R_m * 1e9))
        mostrar(figura_mapas(l, m, real, magnitud, R_nm=R_m * 1e9))
    return {"l": l, "m": m}


def informe_comparacion(l_valor, m_valor, diag=None, dibujar=True):
    diag = diag or Diagnostico()
    titulo_caja("QUANTUM LAB — COMPARACIÓN DE ORBITALES")
    l, m = revisar_lm(l_valor, m_valor, diag)
    if l is not None and l > 60:
        diag.aviso(f"Con l = {entero(l)} los lóbulos son demasiado finos para las figuras pequeñas: se dibujan como "
                   "máximo para l = 60.")
    diag.imprimir()
    if diag.hay_error or l is None or m is None:
        return None
    l_dib = min(l, 60)
    seccion("ORBITALES REALES Y COMPLEJOS")
    vinetas([
        f"Para l = {l} hay {2 * l + 1} orbitales (m = −{l}, …, {l}). Todos tienen la misma energía de rotación.",
        "Complejo Y_l^m: |Y|² no depende de φ (simetría alrededor de z); el color muestra la fase, que gira m veces.",
        "Real: √2·Re o √2·Im de Y_l^|m|. Tiene lóbulos orientados (pₓ, p_y, d_xy…); el color indica el signo.",
        "Y_l^{+m} e Y_l^{−m} tienen la misma forma |Y|² y fases que giran en sentidos opuestos (L_z = ±mħ).",
    ])
    if dibujar:
        mostrar(figura_todos_m(l_dib, real=True))
        if l_dib > 0 and m != 0:
            mostrar(figura_complejo_real(l_dib, m if l == l_dib else max(1, min(abs(m), l_dib))))
        elif l_dib > 0:
            print("\n   (Con m = 0 el orbital complejo y el real coinciden. Elige m ≠ 0 para comparar complejos y reales.)")
    return {"l": l}


def informe_confinada(n_valor, l_valor, R_nm, m_me, diag=None, dibujar=True):
    diag = diag or Diagnostico()
    titulo_caja("QUANTUM LAB — PARTÍCULA CONFINADA EN UNA ESFERA (POZO ESFÉRICO INFINITO)")
    n = revisar_entero(n_valor, "n", diag, "n cuenta las oscilaciones radiales (n − 1 nodos radiales): debe ser entero.")
    if n is not None and n < 1:
        diag.error(f"n = {n} no es válido: n numera los ceros de la función de Bessel (n = 1 es el primero). Con n ≤ 0 "
                   "no hay ningún estado: la función de onda no se anularía en la pared de la esfera.")
        n = None
    l, _ = revisar_lm(l_valor, 0, diag)
    R_m, m_kg = revisar_rotor(R_nm, m_me, diag)
    seccion("PARÁMETROS DEL SISTEMA")
    dato("Número cuántico radial n", entero(n) if n is not None else num(n_valor))
    dato("Momento angular l", entero(l) if l is not None else num(l_valor))
    dato("Radio de la esfera R", num(R_nm), "nm")
    dato("Masa de la partícula", num(m_me), "mₑ")
    if diag.hay_error or n is None or l is None or R_m is None or m_kg is None:
        diag.imprimir()
        return None
    x_nl, metodo = cero_bessel(l, n)
    if metodo.startswith("asintótico"):
        diag.nota(f"Con n = {entero(n)} y l = {entero(l)} el cero de la función de Bessel se calcula con la aproximación "
                  "asintótica de McMahon (muy precisa cuando n ≫ l).")
    log_E = math.log10(HBAR ** 2 / (2 * m_kg * R_m ** 2)) + 2 * math.log10(x_nl)
    revisar_relatividad(log_E, m_kg, diag, f"La energía del estado {n}{letra(l)}")
    if x_nl > 4e4:
        diag.nota("La función radial tiene tantos nodos que no se dibuja; los valores numéricos son exactos.")
    diag.imprimir()
    seccion("RESULTADOS DEL ESTADO")
    dato("Nombre del estado", f"{entero(n)}{letra(l)}")
    dato("Cero de Bessel x_nl (j_l(x_nl) = 0)", num(x_nl, 10), f"({metodo})")
    dato("Energía E = ħ²x²/(2mR²)", num_log(log_E - math.log10(EV)), "eV")
    dato("Energía E", num_log(log_E), "J")
    dato("En unidades de ħ²/(2mR²)", num(x_nl ** 2, 8))
    dato("Nodos radiales (n − 1)", entero(n - 1))
    dato("Degeneración (2l + 1)", entero(2 * l + 1))
    niveles = sorted([(nn, ll, cero_bessel(ll, nn)[0]) for ll in range(0, 5) for nn in range(1, 4)], key=lambda t: t[2])[:10]
    seccion("ORDEN DE LOS NIVELES MÁS BAJOS")
    tabla(["estado", "x_nl", "E [ħ²/2mR²]", "E [eV]", "degeneración"],
          [[("» " if (nn == n and ll == l) else "") + f"{nn}{letra(ll)}", f"{xx:.6f}", f"{xx ** 2:.4f}",
            num(HBAR ** 2 * xx ** 2 / (2 * m_kg * R_m ** 2) / EV), 2 * ll + 1] for nn, ll, xx in niveles])
    seccion("INTERPRETACIÓN FÍSICA")
    vinetas([
        "La función de onda debe anularse en la pared (r = R): eso cuantiza la energía, como en la caja 1D.",
        "A diferencia del átomo de hidrógeno, aquí la energía depende de n y de l: el orden es 1s, 1p, 1d, 2s, 1f…",
        "Este modelo describe bien núcleos atómicos (modelo de capas), puntos cuánticos y moléculas como el C₆₀.",
    ])
    if dibujar:
        mostrar(figura_confinada(n, l, x_nl, R_m, niveles))
    return {"x": x_nl}


def informe_densidad(l_valor, m_valor, real, x_nm, y_nm, z_nm, R_nm, diag=None, dibujar=True):
    """Densidad de probabilidad |Y_l^m|² evaluada en un punto (x, y, z) y graficada frente a la posición cartesiana."""
    diag = diag or Diagnostico()
    titulo_caja("QUANTUM LAB — DENSIDAD DE PROBABILIDAD EN COORDENADAS CARTESIANAS")
    l, m = revisar_lm(l_valor, m_valor, diag)
    R_m, _ = revisar_rotor(R_nm, 1.0, diag)
    seccion("PARÁMETROS")
    dato("Estado (l, mₗ)", f"l = {entero(l) if l is not None else num(l_valor)},  mₗ = "
                            f"{entero(m) if m is not None else num(m_valor)}".replace("-", "−"))
    dato("Función", "orbital real (combinación de ±mₗ)" if real else "armónico complejo Y_l^mₗ (L_z definido)")
    dato("Punto (x, y, z)", f"({num(x_nm)}, {num(y_nm)}, {num(z_nm)})", "nm")
    dato("Radio de la esfera R", num(R_nm), "nm")
    punto = None
    if None not in (x_nm, y_nm, z_nm):
        r, th, ph = a_esfericas(x_nm, y_nm, z_nm)
        if r == 0:
            diag.error("El origen (0, 0, 0) no define ninguna dirección: escribe un punto distinto del origen.")
        else:
            punto = (r, th, ph)
    if l is not None and l > L_DIBUJO_MAX:
        diag.error(f"Con l = {entero(l)} la densidad oscila demasiado para dibujarla (máximo l = {L_DIBUJO_MAX}).")
    if diag.hay_error or l is None or m is None or R_m is None or punto is None:
        diag.imprimir()
        return None
    diag.imprimir()
    real = bool(real and m != 0)
    r, th, ph = punto
    escala = float(np.max(densidad_probabilidad(l, m, *malla_angular(61), real)))
    limpio = lambda v: 0.0 if abs(v) < 1e-12 * escala else v          # quita el ruido de redondeo (≈1e−33) en los nodos
    d = limpio(float(densidad_probabilidad(l, m, th, ph, real)[0]))
    seccion("DEFINICIÓN")
    parrafo("Densidad de probabilidad  P = |Y_l^m|² / R²  [nm⁻²]:  P·dA es la probabilidad de hallar la partícula en el "
            "elemento de área dA = R² sen θ dθ dφ (interpretación de Born).  Densidad angular  |Y_l^m|²  [sr⁻¹].")
    seccion("FÓRMULA")
    parrafo("Coordenadas esféricas del punto: r = √(x² + y² + z²),  cos θ = z/r,  tan φ = y/x.  La partícula está sobre "
            "la esfera de radio R, así que solo importa la dirección (θ, φ).")
    if real:
        parrafo(f"|ψ|² = 2·(2l+1)/(4π)·(l−|m|)!/(l+|m|)!·[P_l^|m|(cos θ)]²·{'cos' if m > 0 else 'sin'}²(|m|φ)   [sr⁻¹]")
    else:
        parrafo("|Y_l^m|² = (2l+1)/(4π)·(l−|m|)!/(l+|m|)!·[P_l^|m|(cos θ)]²   [sr⁻¹]   (no depende de φ)")
    seccion("DENSIDAD EN EL PUNTO ELEGIDO")
    dato("Distancia al origen r", num(r), "nm")
    dato("θ (polar, desde +z)", f"{math.degrees(th):.6f}", "°")
    dato("φ (azimutal, desde +x)", f"{math.degrees(ph):.6f}", "°")
    dato("Punto proyectado sobre la esfera", f"({num(R_nm * x_nm / r)}, {num(R_nm * y_nm / r)}, {num(R_nm * z_nm / r)})", "nm")
    dato("Densidad angular", num(d, 8), "sr⁻¹")
    dato("Densidad de probabilidad", num(d / R_nm ** 2, 8), "nm⁻²")
    if d == 0:
        dato("Observación", "densidad (numéricamente) cero: el punto está sobre una superficie nodal")
    seccion("DENSIDAD EN LOS SEIS PUNTOS DE LOS EJES (sobre la esfera de radio R)")
    ejes = [("+x", (1, 0, 0)), ("−x", (-1, 0, 0)), ("+y", (0, 1, 0)), ("−y", (0, -1, 0)), ("+z", (0, 0, 1)), ("−z", (0, 0, -1))]
    filas = []
    for nombre, (a, b, c) in ejes:
        _, t_, p_ = a_esfericas(a, b, c)
        v = limpio(float(densidad_probabilidad(l, m, t_, p_, real)[0]))
        filas.append([nombre, f"({num(a * R_nm)}, {num(b * R_nm)}, {num(c * R_nm)})", num(v, 6), num(v / R_nm ** 2, 6)])
    tabla(["punto", "(x, y, z) [nm]", "|Y|² [sr⁻¹]", "|Y|²/R² [nm⁻²]"], filas)
    seccion("INTERPRETACIÓN FÍSICA")
    vinetas([
        "|Y|² dΩ es la probabilidad de hallar la partícula en el ángulo sólido dΩ = sinθ dθ dφ; ∫|Y|² dΩ = 1.",
        "El armónico complejo tiene |Y|² con simetría axial: depende solo de z/R = cos θ, no de φ.",
        "Los orbitales reales (pₓ, d_xy…) mezclan +mₗ y −mₗ: tienen lóbulos orientados en x e y, pero no L_z definido.",
        "En los cortes polares la distancia al origen es |Y|² en esa dirección: es una forma de representar la densidad, "
        "no una superficie física.",
    ])
    if dibujar:
        mostrar(figura_densidad_3d(l, m, real, R_nm))
        mostrar(figura_densidad_cartesiana(l, m, real))
    return {"l": l, "m": m, "theta": th, "phi": ph, "densidad": d}


In [ ]:
#@title 5) Pruebas numéricas (comprueban que los cálculos son correctos) { display-mode: "form" }
def ejecutar_pruebas():
    import io, contextlib
    resultados = []

    def prueba(nombre, condicion, detalle=""):
        resultados.append((nombre, bool(condicion), detalle))

    def silencioso(funcion, *args, **kwargs):
        with contextlib.redirect_stdout(io.StringIO()):
            return funcion(*args, **kwargs)

    th = np.array([0.31, 0.9, 1.7, 2.6])
    ph = np.array([0.4, 1.9, 3.3, 5.6])
    s, c = np.sin(th), np.cos(th)
    cerradas = {(0, 0): np.sqrt(1 / (4 * np.pi)) + 0 * th, (1, 0): np.sqrt(3 / (4 * np.pi)) * c,
                (1, 1): -np.sqrt(3 / (8 * np.pi)) * s * np.exp(1j * ph), (1, -1): np.sqrt(3 / (8 * np.pi)) * s * np.exp(-1j * ph),
                (2, 0): np.sqrt(5 / (16 * np.pi)) * (3 * c ** 2 - 1), (2, 1): -np.sqrt(15 / (8 * np.pi)) * s * c * np.exp(1j * ph),
                (2, 2): 0.25 * np.sqrt(15 / (2 * np.pi)) * s ** 2 * np.exp(2j * ph),
                (3, 0): np.sqrt(7 / (16 * np.pi)) * (5 * c ** 3 - 3 * c)}
    for (l, m), y in cerradas.items():
        prueba(f"Y_{l}^{m} = fórmula cerrada (Condon–Shortley)".replace("-", "−"), np.allclose(armonico(l, m, th, ph), y, atol=1e-13))
    for (l, m) in [(0, 0), (3, 2), (10, -7), (40, 13), (300, 150)]:
        nrm = normalizacion_numerica(l, m)
        prueba(f"Normalización ∫|Y|²dΩ = 1 (l = {l}, m = {m})".replace("-", "−"), abs(nrm - 1) < 1e-9, f"{nrm:.12f}")
    x, wq = np.polynomial.legendre.leggauss(40)
    TH, PH = np.meshgrid(np.arccos(x), np.linspace(0, 2 * np.pi, 64, endpoint=False), indexing="ij")
    solap = np.sum(wq[:, None] * np.conj(armonico(2, 1, TH, PH)) * armonico(3, 1, TH, PH)) * 2 * np.pi / 64
    prueba("Ortogonalidad ⟨Y₂¹|Y₃¹⟩ = 0", abs(solap) < 1e-12)
    prueba("p_x real es positivo hacia +x", armonico_real(1, 1, np.array([np.pi / 2]), np.array([0.0]))[0] > 0)
    prueba("p_y real es positivo hacia +y", armonico_real(1, -1, np.array([np.pi / 2]), np.array([np.pi / 2]))[0] > 0)
    prueba("|Y_l^m|² no depende de φ", np.allclose(np.abs(armonico(3, 2, th, ph)) ** 2, np.abs(armonico(3, 2, th, ph + 1.1)) ** 2))
    prueba("Paridad: Y(π−θ, φ+π) = (−1)^l Y", np.allclose(armonico(3, 1, np.pi - th, ph + np.pi), -armonico(3, 1, th, ph)))
    prueba("Conos nodales de Y₃⁰: l − |m| = 3", len(nodos_latitud(3, 0)) == 3, f"{np.round(nodos_latitud(3, 0), 3)}")
    prueba("Cono nodal de p_z en θ = 90°", abs(nodos_latitud(1, 0)[0] - 90) < 1e-8)
    prueba("Planos nodales de d_xy en φ = 0° y 90°", np.allclose(nodos_meridianos(-2), [0, 90]))
    prueba("Primer cero de j₀ = π", abs(cero_bessel(0, 1)[0] - math.pi) < 1e-14)
    prueba("Primer cero de j₁ ≈ 4.4934", abs(cero_bessel(1, 1)[0] - 4.493409457909) < 1e-9, f"{cero_bessel(1, 1)[0]:.10f}")
    prueba("Segundo cero de j₂ ≈ 9.0950", abs(cero_bessel(2, 2)[0] - 9.095011330476) < 1e-9)
    x_num = cero_bessel(3, 4000)[0]
    beta, mu = (4000 + 1.5) * math.pi, 4 * 3.5 ** 2
    prueba("Cero lejano: numérico = McMahon", abs(x_num - (beta - (mu - 1) / (8 * beta))) < 1e-6)
    for etiqueta, args in [("l = 2.5", (2.5, 0)), ("m = 0.5", (2, 0.5)), ("|m| > l", (1, 3))]:
        d = Diagnostico()
        r = silencioso(informe_armonico, *args, True, "densidad", "lobulos", 60, 0.3, 1, 1, diag=d, dibujar=False)
        prueba(f"Dato no físico ({etiqueta}): se explica y no se calcula", r is None and d.hay_error)
    d = Diagnostico()
    r = silencioso(informe_armonico, -3, 1, True, "densidad", "lobulos", 60, 0.3, 1, 1, diag=d, dibujar=False)
    prueba("l = −3 se interpreta como l = 2 (mismo l(l+1))", r is not None and r["l"] == 2)
    d = Diagnostico()
    r = silencioso(informe_armonico, 100000, 5, True, "densidad", "lobulos", 60, 0.3, 1, 1, diag=d, dibujar=False)
    prueba("l = 10⁵: se calcula sin desbordar", r is not None)
    d = Diagnostico()
    r = silencioso(informe_confinada, 0, 0, 1, 1, diag=d, dibujar=False)
    prueba("Confinada con n = 0: se explica", r is None and d.hay_error)
    prueba("Lectura de «2,5» y «1e-30»", leer_numero("2,5")[0] == 2.5 and leer_numero("1e-30")[0] == 1e-30)

    # Densidad de probabilidad en coordenadas cartesianas
    dpz = densidad_probabilidad(1, 0, *a_esfericas(0, 0, 3)[1:])[0]
    prueba("Densidad |Y₁⁰|² en (0, 0, 3) = 3/(4π)", abs(dpz - 3 / (4 * math.pi)) < 1e-12, f"{dpz:.12f}")
    prueba("Densidad |Y₁⁰|² en el plano nodal z = 0 es cero", densidad_probabilidad(1, 0, *a_esfericas(1, 1, 0)[1:])[0] < 1e-25)
    prueba("Densidad |Y₂¹|² a 45° en el plano xz = 15/(32π)",
           abs(densidad_probabilidad(2, 1, *a_esfericas(1, 0, 1)[1:])[0] - 15 / (32 * math.pi)) < 1e-12)
    prueba("Orbital pₓ real: |pₓ|² = 3/(4π)·(x/r)²", abs(densidad_probabilidad(1, 1, *a_esfericas(0.6, 0.0, 0.8)[1:], real=True)[0]
                                                         - 3 / (4 * math.pi) * 0.36) < 1e-12)
    xg, wg = np.polynomial.legendre.leggauss(60)
    TT, PP = np.meshgrid(np.arccos(xg), np.linspace(0, 2 * np.pi, 96, endpoint=False), indexing="ij")
    integral = float(np.sum(wg[:, None] * densidad_probabilidad(3, -2, TT, PP, True).reshape(TT.shape)) * 2 * np.pi / 96)
    prueba("Orbital real normalizado: ∫|ψ|²dΩ = 1 (l = 3, m = −2)", abs(integral - 1) < 1e-10, f"{integral:.12f}")
    h_, v_, d_ = curva_densidad_plano(2, 1, "xz")
    prueba("Corte polar: distancia al origen = |Y|²", np.allclose(np.hypot(h_, v_), d_, atol=1e-14))
    d = Diagnostico()
    r = silencioso(informe_densidad, 1, 0, False, 0, 0, 0, 1, diag=d, dibujar=False)
    prueba("Punto (0, 0, 0): se explica que no define dirección", r is None and d.hay_error)

    titulo_caja("PRUEBAS NUMÉRICAS — PARTÍCULA EN UNA ESFERA")
    tabla(["#", "Comprobación", "Resultado", "Detalle"], alinear="clcl", sangria=1,
          filas=[[i, nombre, "Correcta" if ok else "FALLA", detalle] for i, (nombre, ok, detalle) in enumerate(resultados, 1)])
    correctas = sum(ok for _, ok, _ in resultados)
    print(f"\n   {correctas} de {len(resultados)} pruebas superadas.")
    return correctas, len(resultados)


In [ ]:
#@title 6) Simulador interactivo — partícula en una esfera { display-mode: "form" }
EJEMPLOS = {
    "Orbital p_z (l = 1, m = 0)": {"l": "1", "m": "0", "tipo": False},
    "Orbital d_z² (l = 2, m = 0)": {"l": "2", "m": "0", "tipo": False},
    "Orbital d_xy (l = 2, m = −2)": {"l": "2", "m": "-2", "tipo": True},
    "Orbital f complejo (l = 3, m = 2)": {"l": "3", "m": "2", "tipo": False},
    "l grande: l = 20, m = 10": {"l": "20", "m": "10", "tipo": False},
    "Valor extremo: l = 100000": {"l": "1e5", "m": "3", "tipo": False},
    "Valor no físico: |m| > l": {"l": "1", "m": "3", "tipo": False},
    "Valor no físico: l = 2.5": {"l": "2.5", "m": "0", "tipo": False},
}

entrada_R = Entrada("Radio de la esfera R", 1.0, "nm", tipo="log", rango=(-2, 2))
entrada_masa = Entrada("Masa de la partícula", 1.0, "mₑ (masas del electrón)", tipo="log", rango=(-1, 4))
entrada_l = Entrada("Momento angular l", 2, "", tipo="entero", rango=(0, 10))
entrada_m = Entrada("Número magnético m", 1, "(−l ≤ m ≤ l)", tipo="entero", rango=(-10, 10))
entrada_px = Entrada("Punto: coordenada x", 0.7071, "nm", tipo="real", rango=(-2, 2), paso=0.01)
entrada_py = Entrada("Punto: coordenada y", 0.0, "nm", tipo="real", rango=(-2, 2), paso=0.01)
entrada_pz = Entrada("Punto: coordenada z", 0.7071, "nm", tipo="real", rango=(-2, 2), paso=0.01)
menu_tipo, fila_tipo = selector("Tipo de orbital", [("Complejo (L_z = mħ definido)", False), ("Real (pₓ, d_xy, …)", True)], False)
menu_magnitud, fila_magnitud = selector("Magnitud representada", [(v, k) for k, v in MAGNITUDES.items()], "densidad")
menu_forma, fila_forma = selector("Forma de la superficie", [(v, k) for k, v in FORMAS.items()], "lobulos")
entrada_res = Entrada("Resolución angular (puntos en θ)", 90, "", tipo="entero", rango=(20, 200))
entrada_alfa = Entrada("Amplitud α (deformación suave)", 0.4, "", tipo="real", rango=(0, 0.9), paso=0.05)
entrada_nr = Entrada("Número cuántico radial n", 1, "(n − 1 nodos radiales)", tipo="entero", rango=(1, 10))
entrada_lc = Entrada("Momento angular l", 0, "", tipo="entero", rango=(0, 8))
menu_ejemplo, fila_ejemplo = selector("Configuración de ejemplo", ["— elegir —"] + list(EJEMPLOS), "— elegir —", "420px")

salida_dens, salida_3d, salida_comp, salida_conf, salida_pruebas = w.Output(), w.Output(), w.Output(), w.Output(), w.Output()


def calcular_densidad():
    def tarea():
        d = Diagnostico()
        informe_densidad(entrada_l.leer(d), entrada_m.leer(d), menu_tipo.value, entrada_px.leer(d), entrada_py.leer(d),
                         entrada_pz.leer(d), entrada_R.leer(d), d)
    ejecutar_en(salida_dens, tarea)


def calcular_3d():
    def tarea():
        d = Diagnostico()
        informe_armonico(entrada_l.leer(d), entrada_m.leer(d), menu_tipo.value, menu_magnitud.value, menu_forma.value,
                         entrada_res.leer(d), entrada_alfa.leer(d), entrada_R.leer(d), entrada_masa.leer(d), d)
    ejecutar_en(salida_3d, tarea)


def calcular_comparacion():
    def tarea():
        d = Diagnostico()
        informe_comparacion(entrada_l.leer(d), entrada_m.leer(d), d)
    ejecutar_en(salida_comp, tarea)


def calcular_confinada():
    def tarea():
        d = Diagnostico()
        informe_confinada(entrada_nr.leer(d), entrada_lc.leer(d), entrada_R.leer(d), entrada_masa.leer(d), d)
    ejecutar_en(salida_conf, tarea)


def _aplicar_ejemplo(cambio):
    e = EJEMPLOS.get(cambio["new"])
    if e:
        menu_tipo.value = e["tipo"]
        entrada_m.texto.value = e["m"]
        entrada_l.texto.value = e["l"]


def _al_cambiar_lm():
    calcular_densidad()
    calcular_3d()
    if pestanas.selected_index == 2:
        calcular_comparacion()


menu_ejemplo.observe(_aplicar_ejemplo, "value")
for e in (entrada_l, entrada_m):
    e.al_cambiar(_al_cambiar_lm)
for e in (entrada_res, entrada_alfa):
    e.al_cambiar(calcular_3d)
for menu in (menu_tipo, menu_magnitud, menu_forma):
    menu.observe(lambda _: calcular_3d(), "value")
menu_tipo.observe(lambda _: calcular_densidad(), "value")
for e in (entrada_px, entrada_py, entrada_pz):
    e.al_cambiar(calcular_densidad)
for e in (entrada_R, entrada_masa):
    e.al_cambiar(lambda: (calcular_densidad(), calcular_3d(), calcular_confinada()))
for e in (entrada_nr, entrada_lc):
    e.al_cambiar(calcular_confinada)

pestanas = w.Tab(children=[
    w.VBox([panel("Punto (x, y, z) donde se evalúa la densidad (solo importa su dirección)",
                  [entrada_px.caja, entrada_py.caja, entrada_pz.caja]), salida_dens]),
    w.VBox([panel("Representación", [fila_magnitud, fila_forma, entrada_res.caja, entrada_alfa.caja]), salida_3d]),
    w.VBox([w.HTML("<div style='opacity:.75;margin:4px 0'>Usa l y m del panel «Estado de la partícula».</div>"),
            salida_comp]),
    w.VBox([panel("Estado de la partícula confinada (usa R y la masa de arriba)", [entrada_nr.caja, entrada_lc.caja]),
            salida_conf]),
    w.VBox([boton("Ejecutar las pruebas numéricas", lambda: ejecutar_en(salida_pruebas, ejecutar_pruebas)), salida_pruebas]),
])
for i, t in enumerate(["Densidad de probabilidad", "Armónico 3D", "Comparación", "Esfera confinada", "Pruebas numéricas"]):
    pestanas.set_title(i, t)
pestanas.observe(lambda c: calcular_comparacion() if c["new"] == 2 else None, "selected_index")

encabezado("Partícula en una esfera · armónicos esféricos",
           "Momento angular cuantizado, orbitales en 3D (reales y complejos), superficies nodales, energía de "
           "rotación y partícula confinada dentro de una esfera.")
display(w.VBox([panel("Parámetros del sistema", [ayuda_entradas(), fila_ejemplo, entrada_R.caja, entrada_masa.caja]),
                panel("Estado de la partícula: números cuánticos l y mₗ (se usan en todas las pestañas)",
                      [entrada_l.caja, entrada_m.caja, fila_tipo]),
                pestanas]))
calcular_densidad()
calcular_3d()
calcular_confinada()
